# losses

> Region-overlap and boundary losses for segmentation, written once and valid for binary (`C==1`, sigmoid), multi-class (`C>1`, softmax, mutually exclusive), and multilabel (`C>1`, independent sigmoid per channel, overlap allowed) heads.

A single helper converts raw logits into per-class probabilities and matching targets, so every loss below is class-count and task agnostic. Class-imbalance is handled by Tversky (tunable false-negative vs false-positive weighting); global overlap by Dice; thin-structure edges by a boundary-weighted cross-entropy.

`task` disambiguates the `C > 1` case: `multiclass` assumes classes are mutually exclusive (softmax, one-hot targets); `multilabel` assumes classes can overlap (e.g. `void` inside `chip`) and applies an independent sigmoid per channel instead. See `docs/MULTILABEL_OVERLAP.md` for the full plan.

In [ ]:
#| default_exp losses

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict

import torch
import torch.nn as nn
import torch.nn.functional as F

## Logits -> probabilities + one-hot targets

The one place that knows about binary vs multi-class. Everything downstream sees `(B, C, H, W)` probabilities and a matching `(B, C, H, W)` one-hot target.

In [ ]:
# experiment: shapes for a 2-class softmax case
logits = torch.randn(2, 3, 8, 8)        # B=2, C=3 classes
labels = torch.randint(0, 3, (2, 8, 8)) # integer labels
probs = torch.softmax(logits, 1)
onehot = F.one_hot(labels, 3).permute(0, 3, 1, 2).float()
print('probs', tuple(probs.shape), '| onehot', tuple(onehot.shape))

In [ ]:
#| export
def probs_and_targets(
    logits: torch.Tensor,    # (B, C, H, W) raw logits
    targets: torch.Tensor,   # (B, H, W) labels, or (B, C, H, W) float {0,1} for binary/multilabel, or one-hot
    task: str = None,        # 'binary' | 'multiclass' | 'multilabel'; None infers binary if C==1 else multiclass
) -> tuple:
    """Return `(probs, target_tensor)` both shaped `(B, C, H, W)`.

    `binary` (`C == 1`, the default when `task` is omitted and `C == 1`) and
    `multilabel` (`C > 1`, classes may overlap) both use an independent sigmoid
    per channel and treat `targets` as already `{0,1}` float. `multiclass`
    (`C > 1`, the default when `task` is omitted and `C > 1`) uses softmax over
    channels and one-hot encodes integer-label targets -- classes are mutually
    exclusive, so it must never be used for overlapping labels like `void`
    inside `chip`.
    """
    c = logits.shape[1]
    if task is None:
        task = 'binary' if c == 1 else 'multiclass'
    if task in ('binary', 'multilabel'):
        probs = torch.sigmoid(logits)
        t = targets.float()
        if t.dim() == logits.dim() - 1:
            t = t.unsqueeze(1)
        return probs, t
    probs = torch.softmax(logits, dim=1)
    if targets.dim() == logits.dim():
        return probs, targets.float()
    onehot = F.one_hot(targets.long(), c).permute(0, 3, 1, 2).float()
    return probs, onehot

In [ ]:
# test
p, t = probs_and_targets(torch.randn(2, 1, 4, 4), torch.zeros(2, 4, 4))
assert p.shape == t.shape == (2, 1, 4, 4)
p, t = probs_and_targets(torch.randn(2, 3, 4, 4), torch.randint(0, 3, (2, 4, 4)))
assert p.shape == t.shape == (2, 3, 4, 4)

In [ ]:
# test: multilabel uses independent per-channel sigmoid, not softmax
logits = torch.randn(2, 3, 4, 4)
targets = (torch.rand(2, 3, 4, 4) > 0.5).float()  # already {0,1} per channel, may overlap
p, t = probs_and_targets(logits, targets, task='multilabel')
assert p.shape == t.shape == (2, 3, 4, 4)
assert torch.allclose(t, targets)                       # targets pass through unchanged
assert torch.allclose(p, torch.sigmoid(logits))          # sigmoid, not softmax
assert not torch.allclose(p.sum(1), torch.ones(2, 4, 4))  # channels don't sum to 1 -> not softmax-normalised

# same C, task omitted -> falls back to multiclass (softmax) for backward compatibility
p_mc, _ = probs_and_targets(logits, torch.randint(0, 3, (2, 4, 4)))
assert torch.allclose(p_mc.sum(1), torch.ones(2, 4, 4))
print('probs_and_targets ✓  multilabel = independent sigmoid, multiclass default unchanged')

## Tversky-focal loss

Tversky generalises Dice with separate weights for false negatives (`alpha`) and false positives (`beta`). Raise `alpha` to punish missed targets (recall); raise `beta` to punish false alarms. The focal exponent `1/gamma` focuses on the hardest classes. We reduce per-class then average, so it is correct for any `C`.

In [ ]:
#| export
class TverskyFocalLoss(nn.Module):
    """Per-class Tversky-focal loss for binary, multi-class, or multilabel segmentation.

    Example:
        >>> loss = TverskyFocalLoss()(torch.randn(2, 1, 8, 8), torch.zeros(2, 8, 8))
        >>> loss.ndim
        0
    """

    def __init__(
        self,
        alpha: float = 0.7,   # false-negative weight (recall); higher -> fewer misses
        beta: float = 0.3,    # false-positive weight (precision); higher -> fewer false alarms
        gamma: float = 1.33,  # focal exponent (1/gamma); higher -> focus on hard classes
        smooth: float = 1.0,  # numerical stabiliser
        task: str = None,     # 'binary' | 'multiclass' | 'multilabel'; see `probs_and_targets`
    ):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth
        self.task = task

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p, t = probs_and_targets(logits, targets, task=self.task)
        dims = (0, 2, 3)
        tp = (p * t).sum(dims)
        fn = (t * (1 - p)).sum(dims)
        fp = ((1 - t) * p).sum(dims)
        tversky = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)
        return ((1.0 - tversky) ** (1.0 / self.gamma)).mean()

## Dice loss

Soft Dice gives a stable global-overlap signal that complements Tversky.

In [ ]:
#| export
class DiceLoss(nn.Module):
    "Per-class soft Dice loss (binary, multi-class, or multilabel)."

    def __init__(self, smooth: float = 1.0, task: str = None):
        super().__init__()
        self.smooth = smooth
        self.task = task

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p, t = probs_and_targets(logits, targets, task=self.task)
        dims = (0, 2, 3)
        inter = (p * t).sum(dims)
        dice = (2 * inter + self.smooth) / (p.sum(dims) + t.sum(dims) + self.smooth)
        return 1.0 - dice.mean()

## Boundary loss

Up-weights cross-entropy near target edges, where thin structures are most often lost. The boundary band is found on the GPU with `max_pool2d` (dilation) and its complement (erosion), avoiding a CPU round-trip.

In [ ]:
#| export
class BoundaryLoss(nn.Module):
    "Boundary-weighted cross-entropy (binary, multi-class, or multilabel)."

    def __init__(self, theta0: float = 3.0, theta: float = 5.0, task: str = None):
        super().__init__()
        self.theta0, self.theta = theta0, theta
        self.task = task

    def _weight_map(self, fg: torch.Tensor) -> torch.Tensor:
        "fg: (N, 1, H, W) float {0,1}. Returns per-pixel weights."
        k = int(2 * self.theta0 + 1); pad = k // 2
        dilated = F.max_pool2d(fg, k, 1, pad)
        eroded = 1.0 - F.max_pool2d(1.0 - fg, k, 1, pad)
        boundary = (dilated - eroded).clamp(0, 1)
        return torch.where(boundary > 0, self.theta, 1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        c = logits.shape[1]
        task = self.task or ('binary' if c == 1 else 'multiclass')
        if task in ('binary', 'multilabel'):
            t = targets.float()
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            per_px = F.binary_cross_entropy_with_logits(logits, t, reduction='none')  # (B, C, H, W)
            b, ch, h, w = per_px.shape
            # boundary weighting is per-channel and independent (chip/void/lifted
            # each get their own edge band); fold C into the batch dim for _weight_map
            fg = t.reshape(b * ch, 1, h, w)
            per_px_flat = per_px.reshape(b * ch, 1, h, w)
            return (per_px_flat * self._weight_map(fg)).mean()
        labels = targets.long()
        per_px = F.cross_entropy(logits, labels, reduction='none').unsqueeze(1)
        fg = (labels > 0).float().unsqueeze(1)
        return (per_px * self._weight_map(fg)).mean()

In [ ]:
#| export
class BCELoss(nn.Module):
    """Plain binary cross-entropy for segmentation (binary, multi-class, or multilabel).

    Use instead of `TverskyFocalLoss` when foreground is large relative to the
    image (e.g. a chip fills most of the field of view) and the class-imbalance
    correction of Tversky-focal is not needed.

    Example:
        >>> loss = BCELoss()(torch.randn(2, 1, 8, 8), torch.zeros(2, 8, 8))
        >>> loss.ndim
        0
    """

    def __init__(self, task: str = None):
        super().__init__()
        self.task = task

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        c = logits.shape[1]
        task = self.task or ('binary' if c == 1 else 'multiclass')
        if task in ('binary', 'multilabel'):
            t = targets.float()
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            return F.binary_cross_entropy_with_logits(logits, t)
        return F.cross_entropy(logits, targets.long())


`BCELoss` wraps `F.binary_cross_entropy_with_logits` (binary) and `F.cross_entropy` (multi-class) behind the same interface as the other loss components.

**When to use it:** set `w_bce > 0, w_tversky=0` when the foreground occupies a large fraction of the image — for example a rectangular or rotated-rectangular chip that fills most of the field of view. In that case the class-imbalance correction inside Tversky-focal is unnecessary and plain BCE gives a cleaner gradient signal. The typical pairing is `SegLoss(w_tversky=0, w_dice=0.5, w_bce=0.5)`.

**Test verifies:** scalar output, non-negativity, and gradient flow for binary (C=1) and multi-class (C=3) inputs.


In [ ]:
# test BCELoss: scalar, non-negative, backprops for binary and multi-class
for c, tgt in [(1, torch.zeros(2, 8, 8)), (3, torch.randint(0, 3, (2, 8, 8)))]:
    logits = torch.randn(2, c, 8, 8, requires_grad=True)
    loss = BCELoss()(logits, tgt)
    assert loss.ndim == 0 and float(loss) >= 0, f'C={c}: unexpected loss'
    loss.backward()
    assert logits.grad is not None
print('BCELoss ✓  binary + multi-class')


# Foreground-aware loss

Neuro-JEPA style foreground-aware loss for void detection

In [ ]:
#| export
class ForeGroundAwareLoss:
    
  def __init__(
    self, beta: float=0.1 # weight for background token (0.1 from Neuro-JEPA)
    ):
    self.beta = beta

  def __call__(
            self,
            z: torch.Tensor, # (B,K,D) predicted pixel embeddings for masked token
            h  : torch.Tensor, # (B,K,D) target pixel embeddings for masked token
            chip_mask: torch.Tensor, # (B,1,H,W) Original x-ray intensity
            masked_indices: torch.Tensor, # (B,K) indices of masked tokens
            patch_size: int=16, # patch size for pooling
            ) -> torch.Tensor:
    "Foreground-weighted cross-entropy (binary or multi-class)."
    p2 = torch.quantile(x_ray_patch, 0.02, dim=[2, 3], keepdim=True) # (B,1,1,1)
    p98 = torch.quantile(x_ray_patch, 0.98, dim=[2, 3], keepdim=True) # (B,1,1,1)
    
    # Foreground = within normal solder range,
    # for void detection: void pixels are foreground, background is the region outside chip entirely

    patch_mask = F.avg_pool2d(chip_mask.float(), kernel_size=patch_size, stride=patch_size)

    patch_mask_flat= patch_mask.flatten(1) # (B, num_patches)

    token_fg = torch.gather(
        patch_mask_flat, 1, masked_indices.long()
    )


    # Weight: foreground=1.0, background=beta (0.1 from Neuro-JEPA)
    w = torch.where(
        token_fg > 0.5,
        torch.ones_like(token_fg),
        torch.full_like(token_fg, self.beta)
    
    )

    l1 = torch.abs(z - h).mean(dim=-1) # (B, K)
    loss = (w*l1).sum() / (w.sum() + 1e-8) # weighted average
    return loss

## Hierarchy regularizer (multilabel)

Multilabel channels can have an intended containment hierarchy that nothing
so far encourages during training -- e.g. `void` and `lifted` should stay
inside `chip` (see `docs/MULTILABEL_OVERLAP.md`). `metrics.subset_violation_rate`
*measures* this after the fact; this is the differentiable version used *during*
training: `relu(p_sub - p_super)` is zero wherever the hierarchy already holds
and grows linearly with the violation elsewhere, so it can be added straight
into the loss as a soft nudge (it does not hard-enforce the hierarchy --
`inference.apply_hierarchy` does that at prediction time).

In [ ]:
#| export
def hierarchy_regularizer_loss(
    probs: torch.Tensor,  # (B, C, H, W) sigmoid probabilities (task='multilabel')
    pairs,                 # [(sub_idx, super_idx), ...], e.g. [(1, 0), (2, 0)] for (void,chip),(lifted,chip)
) -> torch.Tensor:
    """Mean `relu(p_sub - p_super)` over all pairs -- 0 where the hierarchy holds, grows with violation."""
    if not pairs:
        return torch.zeros((), device=probs.device)
    terms = [F.relu(probs[:, s] - probs[:, p]).mean() for s, p in pairs]
    return torch.stack(terms).mean()

In [ ]:
# test: zero when hierarchy holds, positive and gradient-bearing when it's violated
probs_ok = torch.zeros(1, 2, 4, 4)
probs_ok[0, 0] = 0.9  # super (chip) high everywhere
probs_ok[0, 1, 1:3, 1:3] = 0.8  # sub (void) high only inside chip's footprint -> no violation
assert float(hierarchy_regularizer_loss(probs_ok, [(1, 0)])) == 0.0

logits_bad = torch.zeros(1, 2, 4, 4, requires_grad=True)
with torch.no_grad():
    logits_bad[0, 1, 0, 0] = 5.0  # sub fires where super (chip) is ~0.5 -> some violation
loss_bad = hierarchy_regularizer_loss(torch.sigmoid(logits_bad), [(1, 0)])
assert float(loss_bad) > 0.0
loss_bad.backward()
assert logits_bad.grad is not None

assert float(hierarchy_regularizer_loss(probs_ok, [])) == 0.0  # no pairs -> no-op
print('hierarchy_regularizer_loss ✓  zero when hierarchy holds, positive + differentiable when violated')

## Combined loss with a phase schedule

The total loss is a weighted sum. Boundary refinement is noisy early on, so a phase schedule starts with coarse shape (Tversky+Dice) and adds boundary weight later.

In [ ]:
#| export
class SegLoss(nn.Module):
    """Weighted Tversky-focal + Dice + Boundary + BCE + hierarchy loss returning a breakdown dict.

    Call `set_phase(1|2|3)` at epoch boundaries to anneal the boundary weight.
    Set `w_bce > 0, w_tversky=0` for a BCE + Dice schedule when foreground is
    large and class-imbalance correction is not needed.

    Set `task='multilabel'` for overlapping classes (e.g. `void` inside `chip`)
    with `C > 1` output channels -- every component loss then uses an
    independent per-channel sigmoid instead of softmax. See `probs_and_targets`.

    Set `w_hierarchy > 0` with `hierarchy_pairs=[(sub_idx, super_idx), ...]` to
    softly nudge sub-channels to stay inside their super-channel during training
    (e.g. `[(1, 0), (2, 0)]` for `(void,chip),(lifted,chip)`). See
    `hierarchy_regularizer_loss`; this does not hard-enforce the hierarchy at
    inference time -- `inference.apply_hierarchy` does that.
    """

    def __init__(
        self,
        alpha: float = 0.7, beta: float = 0.3, gamma: float = 1.33,  # Tversky params
        w_tversky: float = 0.5, w_dice: float = 0.5, w_boundary: float = 0.0,  # component weights
        w_bce: float = 0.0,  # plain BCE weight; set > 0 to enable
        task: str = None,  # 'binary' | 'multiclass' | 'multilabel'; see `probs_and_targets`
        w_hierarchy: float = 0.0,  # hierarchy regularizer weight; set > 0 to enable (multilabel only)
        hierarchy_pairs=None,  # [(sub_idx, super_idx), ...]; required if w_hierarchy > 0
    ):
        super().__init__()
        self.w_tversky, self.w_dice, self.w_boundary, self.w_bce = w_tversky, w_dice, w_boundary, w_bce
        self.w_hierarchy = w_hierarchy
        self.hierarchy_pairs = list(hierarchy_pairs) if hierarchy_pairs else []
        self.task = task
        self.tversky = TverskyFocalLoss(alpha, beta, gamma, task=task)
        self.dice = DiceLoss(task=task)
        self.boundary = BoundaryLoss(task=task)
        self.bce = BCELoss(task=task)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> Dict[str, torch.Tensor]:
        l_tv = self.tversky(logits, targets)
        l_dc = self.dice(logits, targets)
        l_bd = self.boundary(logits, targets) if self.w_boundary > 0 else torch.zeros((), device=logits.device)
        l_bce = self.bce(logits, targets) if self.w_bce > 0 else torch.zeros((), device=logits.device)
        if self.w_hierarchy > 0 and self.hierarchy_pairs:
            l_hier = hierarchy_regularizer_loss(torch.sigmoid(logits), self.hierarchy_pairs)
        else:
            l_hier = torch.zeros((), device=logits.device)
        total = (self.w_tversky * l_tv + self.w_dice * l_dc + self.w_boundary * l_bd
                 + self.w_bce * l_bce + self.w_hierarchy * l_hier)
        return {'loss': total, 'loss_tversky': l_tv.detach(), 'loss_dice': l_dc.detach(),
                'loss_boundary': l_bd.detach(), 'loss_bce': l_bce.detach(), 'loss_hierarchy': l_hier.detach()}

    def set_phase(self, phase: int) -> None:
        "Phase 1: coarse shape; 2: add boundary; 3: polish edges."
        weights = {1: (0.6, 0.4, 0.0), 2: (0.5, 0.3, 0.2), 3: (0.4, 0.3, 0.3)}
        self.w_tversky, self.w_dice, self.w_boundary = weights[phase]

    def set_weights(
        self, w_tversky: float, w_dice: float, w_boundary: float, w_bce: float = 0.0,
        w_hierarchy: float = None,
    ) -> None:
        "Set component weights directly (used by configurable phase schedules)."
        self.w_tversky, self.w_dice, self.w_boundary, self.w_bce = w_tversky, w_dice, w_boundary, w_bce
        if w_hierarchy is not None:
            self.w_hierarchy = w_hierarchy

In [ ]:
# test: every loss is a non-negative scalar and backprops, for binary AND multi-class
for c, tgt in [(1, torch.zeros(2, 16, 16)), (3, torch.randint(0, 3, (2, 16, 16)))]:
    logits = torch.randn(2, c, 16, 16, requires_grad=True)
    crit = SegLoss(w_tversky=0.5, w_dice=0.4, w_boundary=0.1)
    out = crit(logits, tgt)
    assert out['loss'].ndim == 0 and float(out['loss']) >= 0
    assert 'loss_bce' in out and float(out['loss_bce']) == 0.0  # w_bce=0 → zero
    out['loss'].backward()
    assert logits.grad is not None
crit.set_phase(3)
assert crit.w_boundary == 0.3

# BCE + Dice mode: useful when foreground is large (e.g. chip fills the image)
logits = torch.randn(2, 1, 16, 16, requires_grad=True)
crit_bce = SegLoss(w_tversky=0.0, w_dice=0.5, w_boundary=0.0, w_bce=0.5)
out = crit_bce(logits, torch.zeros(2, 16, 16))
assert float(out['loss']) > 0 and float(out['loss_bce']) > 0
out['loss'].backward()
assert logits.grad is not None
print('SegLoss ✓  Tversky+Dice+Boundary | BCE+Dice')

## Phase-scheduled weights, now with `w_hierarchy` (#67)

`SegLoss` already has a hierarchy term, but `train()` never switched it on: the weight
schedule (`resolve_loss_weights`) only knew four weights. The plan, in order:

1. a loss phase may carry its own `w_hierarchy`; a phase without one falls back to a run-wide value (default `0.0`);
2. `resolve_loss_weights` returns that as a 5th entry, so `criterion.set_weights(*weights)` just works;
3. configs that never mention `w_hierarchy` must give the same numbers as before (default off);
4. `validate_hierarchy_loss` fails early, with a readable message, when the term is on but cannot work (not `multilabel`, or no pairs).

In [ ]:
# experiment: what should a phase list with/without w_hierarchy resolve to?
phases = [
    {'until_epoch': 10, 'w_tversky': 0.6, 'w_dice': 0.4, 'w_boundary': 0.0},                      # no own weight
    {'until_epoch': -1, 'w_tversky': 0.5, 'w_dice': 0.3, 'w_boundary': 0.2, 'w_hierarchy': 0.5},  # own weight
]
def _proto(epoch, phases, fallback=0.0):
    ph = next((p for p in phases if p.get('until_epoch', -1) is None or p.get('until_epoch', -1) < 0 or epoch < p['until_epoch']), phases[-1])
    return (float(ph['w_tversky']), float(ph['w_dice']), float(ph['w_boundary']),
            float(ph.get('w_bce', 0.0)), float(ph.get('w_hierarchy', fallback)))
print('epoch 5 :', _proto(5, phases))                  # phase 1 -> fallback 0.0
print('epoch 5 (run-wide 0.3):', _proto(5, phases, 0.3))  # phase 1 -> fallback 0.3
print('epoch 15:', _proto(15, phases, 0.3))            # phase 2 keeps its own 0.5
assert _proto(5, phases)[4] == 0.0 and _proto(5, phases, 0.3)[4] == 0.3 and _proto(15, phases, 0.3)[4] == 0.5

In [ ]:
#| export
def _phase_weights(phase, w_hierarchy):
    return (float(phase['w_tversky']), float(phase['w_dice']), float(phase['w_boundary']),
            float(phase.get('w_bce', 0.0)), float(phase.get('w_hierarchy', w_hierarchy)))

def resolve_loss_weights(
    epoch: int,
    phases,
    w_hierarchy: float = 0.0,  # run-wide fallback for phases that do not set their own `w_hierarchy`
) -> tuple:
    """Return `(w_tversky, w_dice, w_boundary, w_bce, w_hierarchy)` for `epoch` from a phase list.

    `w_bce` and `w_hierarchy` are optional per phase: `w_bce` defaults to `0.0`, `w_hierarchy` to the
    `w_hierarchy` argument (itself `0.0`), so configs that name neither behave exactly as before.
    Feed the tuple to `SegLoss.set_weights(*weights)`.
    """
    for phase in phases:
        until = phase.get('until_epoch', -1)
        if until is None or until < 0 or epoch < until:
            return _phase_weights(phase, w_hierarchy)
    return _phase_weights(phases[-1], w_hierarchy)

In [ ]:
# test: legacy phases resolve unchanged (+ 0.0 hierarchy); per-phase and run-wide hierarchy weights; set_weights round-trip
legacy = [
    {'until_epoch': 10, 'w_tversky': 0.6, 'w_dice': 0.4, 'w_boundary': 0.0},
    {'until_epoch': -1, 'w_tversky': 0.0, 'w_dice': 0.5, 'w_boundary': 0.0, 'w_bce': 0.5},
]
assert resolve_loss_weights(5, legacy) == (0.6, 0.4, 0.0, 0.0, 0.0)
assert resolve_loss_weights(15, legacy) == (0.0, 0.5, 0.0, 0.5, 0.0)
assert resolve_loss_weights(5, phases) == (0.6, 0.4, 0.0, 0.0, 0.0)
assert resolve_loss_weights(5, phases, 0.3)[4] == 0.3          # run-wide fallback
assert resolve_loss_weights(15, phases, 0.3)[4] == 0.5         # phase wins over fallback
crit_w = SegLoss(task='multilabel', hierarchy_pairs=[(2, 0)])
crit_w.set_weights(*resolve_loss_weights(15, phases))
assert crit_w.w_hierarchy == 0.5 and crit_w.w_boundary == 0.2
print('resolve_loss_weights ✓  5-tuple, legacy phases unchanged, per-phase > run-wide > 0.0')

In [ ]:
#| export
def validate_hierarchy_loss(
    task: str,                       # 'binary' | 'multiclass' | 'multilabel'
    hierarchy_pairs,                 # [(sub_idx, super_idx), ...] or None
    phases,                          # loss phase dicts (each may carry `w_hierarchy`)
    w_hierarchy: float = 0.0,        # run-wide fallback weight
    num_classes: int = None,         # when given, pair indices must be < num_classes
) -> bool:
    """Raise `ValueError` if the hierarchy loss is switched on but cannot work; return whether it is on.

    Off (every weight `0`) is always valid, so default configs never hit these checks.
    """
    weights = [float(p.get('w_hierarchy', w_hierarchy)) for p in (phases or [])] or [float(w_hierarchy)]
    if any(w < 0 for w in weights):
        raise ValueError(f'w_hierarchy must be >= 0, got {min(weights)}')
    if not any(w > 0 for w in weights):
        return False
    if task != 'multilabel':
        raise ValueError(
            f"w_hierarchy > 0 needs task='multilabel' (overlapping sigmoid channels), got task={task!r}. "
            'Set train.task: multilabel or train.hierarchy.weight: 0.')
    if not hierarchy_pairs:
        raise ValueError(
            'w_hierarchy > 0 but no hierarchy pairs were resolved. Add e.g. '
            '`train.hierarchy.lifted_subset_of_chip: true` and make sure both names are in data.class_names.')
    if num_classes is not None and any(max(s, p) >= num_classes for s, p in hierarchy_pairs):
        raise ValueError(f'hierarchy_pairs {list(hierarchy_pairs)} index outside num_classes={num_classes}')
    return True

In [ ]:
# test: validate_hierarchy_loss — off is always fine; on needs multilabel + pairs; messages name the fix
assert validate_hierarchy_loss('binary', None, legacy) is False            # default off, any task
assert validate_hierarchy_loss('multiclass', [], legacy, 0.0) is False
assert validate_hierarchy_loss('multilabel', [(2, 0)], phases, 0.0, num_classes=3) is True   # phase 2 has 0.5
assert validate_hierarchy_loss('multilabel', [(2, 0)], legacy, 0.3) is True                  # run-wide weight
for bad, kw in [
    (('multiclass', [(2, 0)], legacy, 0.3), "task='multilabel'"),
    (('binary', [(2, 0)], phases, 0.0), "task='multilabel'"),
    (('multilabel', [], legacy, 0.3), 'no hierarchy pairs'),
    (('multilabel', None, phases, 0.0), 'no hierarchy pairs'),
    (('multilabel', [(2, 0)], legacy, -1.0), '>= 0'),
]:
    try: validate_hierarchy_loss(*bad)
    except ValueError as e: assert kw in str(e), (kw, str(e))
    else: raise AssertionError(f'expected ValueError for {bad}')
try: validate_hierarchy_loss('multilabel', [(5, 0)], legacy, 0.3, num_classes=3)
except ValueError as e: assert 'outside num_classes' in str(e)
else: raise AssertionError('expected out-of-range pair to fail')
print('validate_hierarchy_loss ✓  off passes, non-multilabel / unresolved pairs / bad indices fail with a fix hint')

In [ ]:
# test (#67 wiring): `lifted` predicted outside `chip` costs extra only when w_hierarchy > 0; w=0 / no pairs is bit-identical
torch.manual_seed(0)
tg = torch.zeros(1, 3, 8, 8); tg[0, 0, 2:6, 2:6] = 1; tg[0, 2, 3:5, 3:5] = 1            # chip box, lifted inside it
lg = torch.full((1, 3, 8, 8), -4.0); lg[0, 0, 2:6, 2:6] = 4.0; lg[0, 2] = 4.0           # lifted fires everywhere -> leaks out of chip
mk = lambda **kw: SegLoss(w_tversky=0.5, w_dice=0.5, task='multilabel', **kw)

base = mk()(lg, tg)['loss']
off_zero = mk(w_hierarchy=0.0, hierarchy_pairs=[(2, 0)])(lg, tg)['loss']
off_nopairs = mk(w_hierarchy=1.0, hierarchy_pairs=None)(lg, tg)['loss']
assert torch.equal(base, off_zero) and torch.equal(base, off_nopairs)                    # regression pin: bit-for-bit

lg_on = lg.clone().requires_grad_(True)
on = mk(w_hierarchy=1.0, hierarchy_pairs=[(2, 0)])(lg_on, tg)
assert float(on['loss']) > float(base) and float(on['loss_hierarchy']) > 0
on['loss'].backward()
outside = torch.ones(8, 8, dtype=torch.bool); outside[2:6, 2:6] = False
assert lg_on.grad[0, 2][outside].abs().sum() > 0                                         # gradient reaches lifted logits outside chip

lg_off = lg.clone().requires_grad_(True)
mk(w_hierarchy=0.0, hierarchy_pairs=[(2, 0)])(lg_off, tg)['loss'].backward()
lg_ref = lg.clone().requires_grad_(True)
mk()(lg_ref, tg)['loss'].backward()
assert torch.equal(lg_off.grad, lg_ref.grad)                                             # w=0: same gradients too
print(f"hierarchy wiring ✓  loss {float(base):.4f} -> {float(on['loss']):.4f} with w_hierarchy=1; w=0 identical")

## Multilabel overlap: synthetic test

The whole point of `task='multilabel'` is that classes are allowed to overlap --
`void` sits *inside* `chip`, `lifted` is mostly coextensive with `chip`. Shape
and softmax-guard checks aren't enough evidence this actually works: they'd
also pass for a broken implementation that just ignores the void channel. The
real test is behavioural -- build a synthetic sample where void is a strict
subset of chip, optimize logits directly against `SegLoss(task='multilabel')`,
and check the void channel learns *its own* boundary (high inside the void
region, low in the chip-only ring around it) rather than either collapsing
onto chip or failing to separate from it.

First, a cheap direct sanity check that `BoundaryLoss` and `BCELoss` (which
have their own `task` dispatch, not just a `probs_and_targets` call) accept
3-channel multilabel input and backprop without shape errors -- both were
rewritten above for this and aren't exercised by `SegLoss`'s default
`w_boundary=0.0`.

In [ ]:
# direct sanity check: BoundaryLoss and BCELoss accept multilabel (C=3) input and backprop
logits3 = torch.randn(2, 3, 16, 16, requires_grad=True)
targets3 = (torch.rand(2, 3, 16, 16) > 0.5).float()

l_bd = BoundaryLoss(task='multilabel')(logits3, targets3)
assert l_bd.ndim == 0 and float(l_bd) >= 0
l_bd.backward()
assert logits3.grad is not None

logits3 = torch.randn(2, 3, 16, 16, requires_grad=True)  # fresh leaf for a second backward
l_bce = BCELoss(task='multilabel')(logits3, targets3)
assert l_bce.ndim == 0 and float(l_bce) >= 0
l_bce.backward()
assert logits3.grad is not None
print('BoundaryLoss + BCELoss ✓  multilabel (C=3) shapes and gradients OK')

In [ ]:
# synthetic overlap: void strictly inside chip, lifted mostly coextensive with chip.
# If multilabel training works, the void channel should end up high in the interior
# region and LOW in the chip-only ring around it -- proof it learned its own boundary
# rather than collapsing onto chip (which mutually-exclusive softmax would force).
torch.manual_seed(0)
H = W = 16
chip   = torch.zeros(1, H, W); chip[:, 2:14, 2:14] = 1     # big square
void   = torch.zeros(1, H, W); void[:, 6:10, 6:10] = 1     # small square, strictly inside chip
lifted = torch.zeros(1, H, W); lifted[:, 3:13, 3:13] = 1   # mostly coextensive with chip
targets = torch.stack([chip, void, lifted], dim=1)  # (1, 3, H, W)

assert bool((void.bool() & ~chip.bool()).sum() == 0), 'void must be a subset of chip'
assert bool((lifted.bool() & ~chip.bool()).sum() == 0), 'lifted must be a subset of chip'

logits = torch.zeros(1, 3, H, W, requires_grad=True)
crit = SegLoss(w_tversky=0.5, w_dice=0.5, task='multilabel')
opt = torch.optim.Adam([logits], lr=0.5)

losses = []
for _ in range(150):
    opt.zero_grad()
    out = crit(logits, targets)
    out['loss'].backward()
    opt.step()
    losses.append(float(out['loss']))

assert losses[-1] < losses[0] * 0.1, f'loss did not converge: {losses[0]:.4f} -> {losses[-1]:.4f}'

probs = torch.sigmoid(logits.detach())
void_interior  = probs[0, 1, 7, 7].item()   # inside void region
void_chip_only = probs[0, 1, 3, 3].item()   # inside chip, outside void
chip_interior  = probs[0, 0, 7, 7].item()   # chip fires at the same pixel too (overlap)

assert void_interior > 0.5,  f'void channel failed to learn interior void: {void_interior:.3f}'
assert void_chip_only < 0.5, f'void channel leaked into chip-only region: {void_chip_only:.3f}'
assert chip_interior > 0.5,  f'chip channel should also fire at the overlap pixel: {chip_interior:.3f}'
print(f'synthetic overlap ✓  loss {losses[0]:.4f} -> {losses[-1]:.4f} | '
      f'void@interior={void_interior:.3f} void@chip-only={void_chip_only:.3f} chip@interior={chip_interior:.3f}')

In [ ]:
# test: SegLoss's hierarchy term alone pulls a leaking sub-channel back inside its super-channel.
# w_tversky=w_dice=0 so *only* the hierarchy regularizer can move the logits -- isolates its effect.
torch.manual_seed(0)
logits_h = torch.zeros(1, 2, 8, 8, requires_grad=True)
with torch.no_grad():
    logits_h[0, 0, 2:6, 2:6] = 4.0   # super (chip): confidently on in a 4x4 box
    logits_h[0, 1, 0:8, 0:8] = 4.0   # sub (void): confidently on everywhere -- violates outside the box
targets_h = torch.zeros(1, 2, 8, 8)  # irrelevant: w_tversky=w_dice=0

crit_h = SegLoss(w_tversky=0.0, w_dice=0.0, w_hierarchy=1.0, hierarchy_pairs=[(1, 0)], task='multilabel')
opt_h = torch.optim.Adam([logits_h], lr=0.5)

before = float(hierarchy_regularizer_loss(torch.sigmoid(logits_h.detach()), [(1, 0)]))
out_h = None
for _ in range(100):
    opt_h.zero_grad()
    out_h = crit_h(logits_h, targets_h)
    out_h['loss'].backward()
    opt_h.step()
after = float(hierarchy_regularizer_loss(torch.sigmoid(logits_h.detach()), [(1, 0)]))

assert after < before * 0.1, f'hierarchy regularizer did not reduce violation: {before:.4f} -> {after:.4f}'
assert 'loss_hierarchy' in out_h and float(out_h['loss_hierarchy']) >= 0
print(f'SegLoss hierarchy regularizer ✓  violation {before:.4f} -> {after:.4f} (only w_hierarchy active, no supervision)')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()